# 05. Multimodal fusion: cross-attention (Phase 5)

Cross-attention giữa token text (PhoBERT, tối đa 256 token) và 50 token ảnh (CLIP ViT-B/32: 1 CLS + 49 patch 7x7). Mỗi chiều là một khối multi-head attention (8 head, hidden 512) + FFN + LayerNorm. Biểu diễn sau attention được pool rồi nối thành `[t; i; t*i; |t-i|]` (2048 chiều), qua MLP `2048 → 512 → 4`.

Plan Phase 5 chạy **riêng từng chiều** rồi so với concat của Phase 4. Mọi config ở đây đóng băng cả hai encoder, chỉ train khối cross-attention và head, cùng chế độ với `fusion_concat_frozen`, nên chênh lệch đến từ cách kết hợp modality.

| Config | Chiều attention | Task plan | Vai trò |
|---|---|---|---|
| `text_only_frozen` | — | mốc | PhoBERT đóng băng + MLP (từ Phase 4) |
| `image_only_frozen` | — | mốc | CLIP đóng băng + MLP (từ Phase 4) |
| `fusion_concat_frozen` | — | mốc | concat + MLP (Phase 4) |
| `fusion_cross_attn_t2i_frozen` | text attend vào ảnh | 5.1, 5.3 | token text là query, patch ảnh là key/value |
| `fusion_cross_attn_i2t_frozen` | ảnh attend vào text | 5.2, 5.4 | patch ảnh là query, token text là key/value |
| `fusion_cross_attn_frozen` | cả hai chiều | tham khảo | mặc định của `CrossAttentionFusion` |

Mọi config: tối đa 20 epoch, early stopping patience 3 theo val macro-F1, CE, sampler random, batch 16, `lr_head` 1e-3. Chiều không chạy thì ảnh (hoặc text) chỉ qua phép chiếu tuyến tính trước khi pool, nên `out_dim` giống nhau ở cả ba chế độ.

Train = toàn bộ tập train (10.805), val = public test (1.413), test = private test (1.504) (`data.split: official`). Mỗi config chạy 3 seed (`SEEDS`), seed 42 trùng các phase trước.

**Đọc macro-F1 cẩn thận:** test chỉ có 7 mẫu `text-sarcasm`, đúng 3/7 hay 0/7 làm macro-F1 4 lớp lệch khoảng 0.15. Bảng có thêm macro-F1 3 lớp (bỏ `text-sarcasm`) và weighted-F1.

Kaggle: bật **Internet** và **GPU T4 x2** (1 GPU vẫn chạy được, chậm gấp đôi). Add Input:
1. `phuclkz/vimmsd-private-dataset`: ảnh và nhãn gốc (public/private test có nhãn). **Bắt buộc**, cross-attention cần ảnh.
2. `vtnmai/vimmsd-data-preprocessed`: `text_preprocessed.json` (caption đã tách từ). `scripts/train.py` tự copy sang cache.
3. (khuyến nghị) output version của notebook 04 có các thư mục `text_only_frozen_s*`, `image_only_frozen_s*`, `fusion_concat_frozen_s*`: đặt `PREVIOUS_OUTPUT` ở cell cấu hình để dùng lại 3 mốc, chỉ train 3 config cross-attention (9 run).
4. (tuỳ chọn) output version trước của chính notebook này, để chạy tiếp khi session bị ngắt (cũng qua `PREVIOUS_OUTPUT`).

Thời gian ước tính trên 2 GPU: khoảng 10-15 phút mỗi run cross-attention, 9 run chạy 2 song song.
Output trong `/kaggle/working/outputs/<config>_s<seed>/`: `best.pt`, `config.json`, `test_metrics.json`, `best_val_metrics.json`, `confusion_matrix_test.png`, `test_predictions.csv`, `results_row.md`, `run.log`.

In [ ]:
# === Cell bootstrap (giống nhau ở mọi notebook, xem CODEBASE.md mục 6) ===
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/meo225/Multimodal_Sarcasm_Detection.git"
REF = "main"               # branch của mình, hoặc 1 commit hash để tái lặp kết quả (ghi hash vào reports/results.md)
CODE_DIR = "/tmp/IE403"    # clone ra ngoài /kaggle/working để token không bị lưu vào output


def get_github_token():
    """Token cho repo private; repo public thì không cần tạo secret."""
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    if "COLAB_RELEASE_TAG" in os.environ and not os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):  # ảnh Kaggle dựng từ ảnh Colab nên cũng có biến này
        from google.colab import drive
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")  # kernel đang chạy chưa thấy package vừa cài -e
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # local: chạy từ gốc repo (đã `pip install -e .` một lần)

print("cwd:", os.getcwd())
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", commit or "unknown")

In [ ]:
# Chạy test của repo trước khi train: vài chục giây, model trong test là bản tí hon nên không tải weight.
# Test fail thì dừng notebook luôn, không tốn GPU train code lỗi. Test cần Java/VnCoreNLP tự skip nếu thiếu.
RUN_TESTS = True
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
    result = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-x", "-p", "no:cacheprovider"],
                            capture_output=True, text=True)
    print(result.stdout[-5000:], result.stderr[-2000:])
    if result.returncode != 0:
        raise RuntimeError("pytest fail, xem output phía trên trước khi train")

In [ ]:
import gc
import json
import logging
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from vimmsd.utils.config import load_config

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)

# Bỏ bớt config / seed nếu chỉ muốn chạy lại một phần. Vòng ngoài là seed: bị ngắt giữa chừng vẫn có đủ 6 config của seed đầu.
RUNS = ["text_only_frozen", "image_only_frozen", "fusion_concat_frozen",
        "fusion_cross_attn_t2i_frozen", "fusion_cross_attn_i2t_frozen", "fusion_cross_attn_frozen"]
SEEDS = [42, 1, 2]   # 42 = seed của Phase 3
SPLIT_SEED = 42      # cố định train/val/test cho mọi seed
# True: run đã có test_metrics.json và test_predictions.csv thì bỏ qua (attach output version trước rồi chạy tiếp)
SKIP_DONE = True
# Thư mục outputs/ của output version trước (Add Input), vd "/kaggle/input/<tên output notebook 04 hoặc 05>/outputs".
# Chạy lại trong cùng session (kết quả đã ở /kaggle/working/outputs) thì để None.
# Các run đã xong được copy sang OUT (bỏ file .pt) để SKIP_DONE bỏ qua. None = không dùng.
PREVIOUS_OUTPUT = None
RARE = "text-sarcasm"  # lớp chỉ có 7 mẫu test, tách riêng khi đọc macro-F1


def run_name(name, seed):
    return f"{name}_s{seed}"


cfgs = {name: load_config(f"configs/{name}.yaml") for name in RUNS}
OUT = Path(next(iter(cfgs.values())).paths.output_dir)
LABELS = list(next(iter(cfgs.values())).data.labels)
print("output:", OUT)
print("labels:", LABELS)
print({name: (cfg.train.epochs, cfg.train.lr_head, cfg.train.lr_encoder) for name, cfg in cfgs.items()})

## Kiểm tra data

Báo lỗi sớm nếu thiếu dataset, thay vì đợi tới lúc train. `stage_preprocessed_cache` là hàm `train.py` cũng gọi: copy `text_preprocessed.json` từ `/kaggle/input` sang `cache_dir` ghi được.

In [ ]:
from vimmsd.data.dataset import stage_preprocessed_cache

cfg0 = next(iter(cfgs.values()))
data_dir = Path(cfg0.paths.data_dir)
if not (data_dir / cfg0.data.train_json).exists():
    raise FileNotFoundError(f"không thấy {data_dir / cfg0.data.train_json}: Add Input phuclkz/vimmsd-private-dataset")
for path in stage_preprocessed_cache(cfg0):
    meta = json.loads(path.read_text(encoding="utf-8")).get("meta", {})
    print(f"{path.name}: pipeline_key={meta.get('pipeline_key')}, preprocess_version={meta.get('preprocess_version')}")

from collections import Counter

from vimmsd.data.dataset import load_all_records

splits = load_all_records(cfg0)
for split in ("train", "val", "test"):
    recs = splits[split]
    missing = sum(not Path(r["image_path"]).exists() for r in recs)
    print(f"{split}: {len(recs)} mẫu, thiếu ảnh {missing} | nhãn {dict(Counter(cfg0.data.labels[r['label']] for r in recs))}")
assert (len(splits["train"]), len(splits["val"]), len(splits["test"])) == (10805, 1413, 1504), "số mẫu không khớp chia theo cuộc thi"


## Số parameters train được

Build từng model trên CPU để đếm, rồi giải phóng. Dùng cho bảng thống kê tài nguyên.

In [ ]:
from vimmsd.models.classifier import build_model

param_rows = []
for name, cfg in cfgs.items():
    model = build_model(cfg)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    head = sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and not n.startswith(("text_encoder.", "image_encoder.")))
    param_rows.append({"config": name, "trainable (M)": trainable / 1e6, "trong đó encoder (M)": (trainable - head) / 1e6,
                       "head (M)": head / 1e6, "tổng (M)": total / 1e6})
    del model
    gc.collect()
params = pd.DataFrame(param_rows).set_index("config")
params.round(2)

## Train + đánh giá

Mỗi config × seed: `train.py` (train, chọn best checkpoint theo val macro-F1, chấm test) rồi `evaluate.py --split test` (ghi `test_predictions.csv` để phân tích lỗi).
Override chung: `seed=<seed> data.split_seed=42 name=<config>_s<seed>`.

**2 GPU:** mỗi run chỉ dùng khoảng 1-1,5 GB VRAM nên không chia một run ra 2 GPU (DDP), mà chạy 2 run song song, mỗi GPU một run (`vimmsd.utils.gpu_queue`, `CUDA_VISIBLE_DEVICES=k`).
Log đầy đủ của từng run nằm ở `outputs/<run>/run.log`; cell chỉ in các dòng epoch khi run xong.

**Dung lượng:** `/kaggle/working` giới hạn khoảng 20 GB. Checkpoint chỉ lưu parameter được train (bỏ encoder đóng băng, tải lại từ pretrained khi load), và `last.pt` bị xoá khi run xong, nên mỗi run đóng băng chỉ tốn vài MB; `fusion_concat_ftlast` khoảng 120 MB.
Hai run dùng chung 4 CPU của Kaggle (mỗi run 2 worker load ảnh) nên `train (phút)` của từng run có thể cao hơn khi chạy một mình; khi so tài nguyên giữa các config, chỉ so các run chạy cùng chế độ.

Session bị ngắt: đặt `PREVIOUS_OUTPUT` trỏ vào output version trước rồi chạy lại cell này, các run đã xong được bỏ qua.

In [ ]:
import shutil

import torch

from vimmsd.utils.gpu_queue import Job, run_on_gpus

NUM_GPUS = torch.cuda.device_count()  # Kaggle T4 x2 -> 2 run song song
print("GPU:", NUM_GPUS, [torch.cuda.get_device_name(k) for k in range(NUM_GPUS)])

if PREVIOUS_OUTPUT and Path(PREVIOUS_OUTPUT).resolve() == OUT.resolve():
    print("PREVIOUS_OUTPUT trùng OUT: kết quả đã nằm sẵn ở đây, không cần copy")
elif PREVIOUS_OUTPUT:
    if not Path(PREVIOUS_OUTPUT).is_dir():
        raise FileNotFoundError(f"không thấy PREVIOUS_OUTPUT={PREVIOUS_OUTPUT}, xem đường dẫn ở panel Input")
    for metrics in sorted(Path(PREVIOUS_OUTPUT).glob("*/test_metrics.json")):
        shutil.copytree(metrics.parent, OUT / metrics.parent.name, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("*.pt"))
        print("dùng lại", metrics.parent.name)


def make_job(name, seed):
    run = run_name(name, seed)
    out_dir = OUT / run
    overrides = [f"seed={seed}", f"data.split_seed={SPLIT_SEED}", f"name={run}"]
    train = [sys.executable, "scripts/train.py", "--config", f"configs/{name}.yaml", "--override", *overrides,
             "--note", f"Phase 5 {name} seed {seed}"]
    evaluate = [sys.executable, "scripts/evaluate.py", "--config", f"configs/{name}.yaml", "--split", "test",
                "--override", *overrides]

    def check():
        missing = [f for f in ("test_metrics.json", "test_predictions.csv") if not (out_dir / f).exists()]
        if missing:
            return f"thiếu {', '.join(missing)}"
        (out_dir / "last.pt").unlink(missing_ok=True)  # chỉ cần để resume giữa run; run xong thì bỏ cho đỡ tốn đĩa
        return None

    return Job(run, [train, evaluate], out_dir / "run.log", check=check)


jobs = []
for seed in SEEDS:  # vòng ngoài là seed: hàng đợi chạy hết seed đầu trước
    for name in RUNS:
        out_dir = OUT / run_name(name, seed)
        if SKIP_DONE and (out_dir / "test_metrics.json").exists() and (out_dir / "test_predictions.csv").exists():
            print(f"=== bỏ qua {run_name(name, seed)}: đã có kết quả trong {out_dir}")
            # run cũ (trước khi checkpoint bỏ encoder đóng băng) còn last.pt gần 1 GB, file .tmp ghi dở khi hết đĩa
            for f in [out_dir / "last.pt", *out_dir.glob("*.tmp")]:
                f.unlink(missing_ok=True)
            continue
        # run chưa xong (lỗi giữa chừng) train lại từ đầu: xoá thư mục cũ, có thể còn checkpoint .tmp ghi dở vài trăm MB
        shutil.rmtree(out_dir, ignore_errors=True)
        jobs.append(make_job(name, seed))
print(f"=== {len(jobs)} run, {max(NUM_GPUS, 1)} run cùng lúc")
OUT.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUT).free / 1e9
print(f"đĩa còn trống: {free_gb:.1f} GB")
if jobs and free_gb < 1:
    raise RuntimeError("đĩa gần đầy: xoá bớt file trong /kaggle/working (!du -sh /kaggle/working/*) trước khi train")

finished = run_on_gpus(jobs, NUM_GPUS, summary_pattern=r"INFO epoch|early stopping|^\| \d{4}-")
print(f"đĩa còn trống: {shutil.disk_usage(OUT).free / 1e9:.1f} GB")
failed = [job for job in finished if job.error]
if failed:
    raise RuntimeError("run lỗi: " + ", ".join(f"{job.name} ({job.error}, log {job.log})" for job in failed))

## Bảng so sánh

Mỗi dòng ở bảng chi tiết là một run (config × seed). Bảng tổng hợp lấy trung bình ± độ lệch chuẩn qua các seed.
- `val`: best val macro-F1 (checkpoint được chọn); `test`: trên tập test chia từ train.
- `macro-F1 (3 lớp)`: bỏ `text-sarcasm` (7 mẫu test), ít nhiễu hơn macro-F1 4 lớp.
- Thời gian, peak VRAM đọc từ `results_row.md`.

Baseline Phase 3 (fine-tune toàn bộ, 1 seed) được thêm vào nếu tìm thấy trong `/kaggle/input`, hoặc điền tay vào `MANUAL_REFERENCE`.

In [ ]:
# Baseline Phase 3 nhập tay khi không attach được output, ví dụ:
# MANUAL_REFERENCE = {"image_only (Phase 3)": {"macro_f1": 0.3945}}
MANUAL_REFERENCE = {}


def parse_results_row(path):
    """| ngày | `config` | `commit` | env | val | test | phút | ghi chú | -> phút, peak VRAM, commit."""
    if not path.exists():
        return {}
    line = path.read_text(encoding="utf-8").strip().splitlines()[-1]  # chạy lại thì dòng mới nhất ở cuối
    cols = [c.strip() for c in line.strip("|").split("|")]  # ghi chú có thể chứa "|" nên chỉ dùng 7 cột đầu
    vram = re.search(r"peak VRAM ([\d.]+) GB", line)
    return {"commit": cols[2].strip("`"), "train (phút)": float(cols[6]),
            "peak VRAM (GB)": float(vram.group(1)) if vram else None}


def metrics_row(config, seed, out_dir, mode):
    test = json.loads((out_dir / "test_metrics.json").read_text())
    val_path = out_dir / "best_val_metrics.json"
    per_class = test["per_class"]
    row = {"config": config, "seed": seed, "chế độ": mode,
           "val macro-F1": json.loads(val_path.read_text())["macro_f1"] if val_path.exists() else np.nan,
           "test macro-F1": test["macro_f1"],
           "test macro-F1 (3 lớp)": np.mean([m["f1"] for c, m in per_class.items() if c != RARE]),
           "test weighted-F1": test["weighted_f1"],
           "test macro-P": test["macro_precision"], "test macro-R": test["macro_recall"], "accuracy": test["accuracy"]}
    row.update({f"F1 {c}": m["f1"] for c, m in per_class.items()})
    row.update(parse_results_row(out_dir / "results_row.md"))
    return row


MODES = {"text_only_frozen": "đóng băng", "image_only_frozen": "đóng băng",
         "fusion_concat_frozen": "đóng băng, concat",
         "fusion_cross_attn_t2i_frozen": "đóng băng, text attend ảnh",
         "fusion_cross_attn_i2t_frozen": "đóng băng, ảnh attend text",
         "fusion_cross_attn_frozen": "đóng băng, 2 chiều"}
rows = [metrics_row(name, seed, OUT / run_name(name, seed), MODES[name])
        for seed in SEEDS for name in RUNS if (OUT / run_name(name, seed) / "test_metrics.json").exists()]

for ref in ("text_only", "image_only"):
    hits = sorted(Path("/kaggle/input").rglob(f"{ref}/test_metrics.json")) if Path("/kaggle/input").exists() else []
    if hits:
        rows.append(metrics_row(f"{ref} (Phase 3)", 42, hits[0].parent, "fine-tune toàn bộ"))
for label, m in MANUAL_REFERENCE.items():
    rows.append({"config": label, "seed": 42, "chế độ": "fine-tune toàn bộ", "test macro-F1": m.get("macro_f1")})

runs = pd.DataFrame(rows)
pd.set_option("display.max_columns", None)
print("Chi tiết từng run:")
display(runs.set_index(["config", "seed"]).round(4))

In [ ]:
METRICS = ["val macro-F1", "test macro-F1", "test macro-F1 (3 lớp)", "test weighted-F1", "accuracy",
           *[f"F1 {c}" for c in LABELS]]
RESOURCES = ["train (phút)", "peak VRAM (GB)"]
order = [c for c in [*RUNS, "text_only (Phase 3)", "image_only (Phase 3)", *MANUAL_REFERENCE] if c in set(runs["config"])]
grouped = runs.groupby("config", sort=False)
mean = grouped[METRICS + RESOURCES].mean().reindex(order)
std = grouped[METRICS].std().reindex(order)  # NaN khi chỉ có 1 seed
n_seeds = grouped.size().reindex(order)

summary = pd.DataFrame(index=order)
summary["chế độ"] = grouped["chế độ"].first().reindex(order)
summary["số seed"] = n_seeds
for col in METRICS:
    summary[col] = [f"{m:.3f}" if np.isnan(s) else f"{m:.3f} ± {s:.3f}" for m, s in zip(mean[col], std[col])]
for col in RESOURCES:
    summary[col] = mean[col].round(2)
summary = summary.join(params[["trainable (M)"]].round(2), how="left")
print("Trung bình ± độ lệch chuẩn qua các seed:")
display(summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
x = np.arange(len(order))
for offset, col in ((-0.2, "test macro-F1"), (0.2, "test macro-F1 (3 lớp)")):
    axes[0].bar(x + offset, mean[col], 0.4, yerr=std[col].fillna(0), capsize=3, label=col)
axes[0].set_xticks(x, order, rotation=20, ha="right")
axes[0].set_ylabel("F1")
axes[0].set_title("Macro-F1 trên test (trung bình ± std qua seed)")
axes[0].legend(fontsize=8)

width = 0.8 / len(order)
for k, config in enumerate(order):
    cols = [f"F1 {c}" for c in LABELS]
    axes[1].bar(np.arange(len(LABELS)) + (k - (len(order) - 1) / 2) * width, mean.loc[config, cols],
                width, yerr=std.loc[config, cols].fillna(0), capsize=2, label=config)
axes[1].set_xticks(np.arange(len(LABELS)), LABELS)
axes[1].set_ylabel("F1")
axes[1].set_title("F1 từng lớp")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Ma trận nhầm lẫn (test, seed đầu tiên)

In [ ]:
done = [name for name in RUNS if (OUT / run_name(name, SEEDS[0]) / "confusion_matrix_test.png").exists()]
fig, axes = plt.subplots(1, len(done), figsize=(5 * len(done), 4.5), squeeze=False)
for ax, name in zip(axes[0], done):
    ax.imshow(plt.imread(OUT / run_name(name, SEEDS[0]) / "confusion_matrix_test.png"))
    ax.set_title(run_name(name, SEEDS[0]))
    ax.axis("off")
plt.tight_layout()
plt.show()

## Diễn biến train (từng epoch)

Đọc từ `run.log` của từng run (seed đầu tiên). Cần thấy: train loss giảm đều; val macro-F1 đạt đỉnh rồi giảm hoặc đi ngang (nơi early stopping dừng). Train loss tụt rất nhanh trong khi val không tăng là dấu hiệu học thuộc.

In [ ]:
EPOCH_RE = re.compile(r"epoch (\d+) \| train loss ([\d.]+) \| val chọn checkpoint \([^)]*\) ([\d.]+) \| val macro-F1 ([\d.]+) \| macro-P ([\d.]+) \| macro-R ([\d.]+) \| acc ([\d.]+)")


def read_history(name, seed):
    log = OUT / run_name(name, seed) / "run.log"
    if not log.exists():
        return pd.DataFrame()
    rows = [dict(zip(["epoch", "train loss", "val select", "val macro-F1", "val macro-P", "val macro-R", "val acc"],
                     map(float, m.groups()))) for m in map(EPOCH_RE.search, log.read_text(encoding="utf-8").splitlines()) if m]
    return pd.DataFrame(rows)


histories = {name: read_history(name, SEEDS[0]) for name in RUNS}
histories = {name: h for name, h in histories.items() if len(h)}
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name, h in histories.items():
    axes[0].plot(h["epoch"], h["train loss"], marker="o", label=name)
    axes[1].plot(h["epoch"], h["val macro-F1"], marker="o", label=name)
    axes[2].plot(h["epoch"], h["val acc"], marker="o", label=name)
for ax, title in zip(axes, ("train loss", "val macro-F1", "val accuracy")):
    ax.set_title(f"{title} (seed {SEEDS[0]})"); ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
best_epochs = pd.DataFrame({name: {"epoch tốt nhất": int(h.loc[h["val macro-F1"].idxmax(), "epoch"]),
                                    "số epoch đã chạy": int(h["epoch"].max()),
                                    "val macro-F1 tốt nhất": h["val macro-F1"].max(),
                                    "train loss cuối": h["train loss"].iloc[-1]} for name, h in histories.items()}).T
display(best_epochs.round(4))
for name in ("fusion_cross_attn_t2i_frozen", "fusion_cross_attn_i2t_frozen"):
    if name in histories:
        print(f"--- {name}: từng epoch (seed {SEEDS[0]})")
        display(histories[name].set_index("epoch").round(4))

## Cross-attention giúp được gì? So sánh từng mẫu với mốc

Mỗi cặp (mốc, fusion) cùng chế độ đóng băng, cùng loss và cùng seed, nên khác biệt chỉ đến từ cách kết hợp. Với mỗi seed, đếm mẫu fusion đúng mà mốc sai ("giúp") và ngược lại ("nhiễu") theo nhãn thật, rồi lấy trung bình qua các seed.
- Mốc `text_only_frozen`: ảnh (qua cross-attention) thêm được gì so với chỉ dùng caption.
- Mốc `fusion_concat_frozen`: cross-attention hơn nối vector thẳng ở những mẫu nào.

In [ ]:
def load_pred(name, seed):
    return pd.read_csv(OUT / run_name(name, seed) / "test_predictions.csv").set_index("id")


HELP, HURT = "giúp (fusion đúng, mốc sai)", "nhiễu (mốc đúng, fusion sai)"
examples = {}
PAIRS = [("text_only_frozen", "fusion_cross_attn_t2i_frozen"), ("text_only_frozen", "fusion_cross_attn_i2t_frozen"),
         ("text_only_frozen", "fusion_cross_attn_frozen"), ("fusion_concat_frozen", "fusion_cross_attn_t2i_frozen"),
         ("fusion_concat_frozen", "fusion_cross_attn_i2t_frozen"), ("fusion_concat_frozen", "fusion_cross_attn_frozen")]
for text_name, fusion_name in PAIRS:
    per_seed = []
    for seed in SEEDS:
        if not all((OUT / run_name(n, seed) / "test_predictions.csv").exists() for n in (text_name, fusion_name)):
            continue
        both = load_pred(text_name, seed)[["label", "pred", "caption"]].join(
            load_pred(fusion_name, seed)[["pred"]], rsuffix="_fusion")
        base_ok, fusion_ok = both["pred"] == both["label"], both["pred_fusion"] == both["label"]
        both["mốc đúng"], both["fusion đúng"] = base_ok, fusion_ok
        both[HELP], both[HURT] = fusion_ok & ~base_ok, base_ok & ~fusion_ok
        per_seed.append(both.groupby("label")[["mốc đúng", "fusion đúng", HELP, HURT]].sum())
        examples.setdefault(fusion_name, both)
    if per_seed:
        summary_pairs = sum(per_seed) / len(per_seed)
        summary_pairs.insert(0, "n", examples[fusion_name].groupby("label").size())
        summary_pairs.loc["tổng"] = summary_pairs.sum()
        print(f"=== {text_name} vs {fusion_name} (trung bình {len(per_seed)} seed)")
        display(summary_pairs.round(1))

In [ ]:
# Vài mẫu cross-attention giúp so với text_only_frozen ở seed đầu tiên: cross-attention đúng, mốc sai
for name in ("fusion_cross_attn_t2i_frozen", "fusion_cross_attn_i2t_frozen"):
    both = examples.get(name)
    if both is not None:
        helped = both[both[HELP]]
        print(f"=== {name}: {len(helped)} mẫu {name} đúng, mốc sai (mốc là text_only_frozen)")
        display(helped.sample(min(10, len(helped)), random_state=0)[["label", "pred", "pred_fusion", "caption"]])

## Attention map (nghiệm thu bằng mắt)

Lấy `best.pt` của `fusion_cross_attn_frozen_s<seed đầu>` (cả hai chiều), vẽ trên 6 mẫu test `image-sarcasm` / `multi-sarcasm`:
- Cột 2: attention của token text lên từng patch ảnh (7x7, phóng lên ảnh gốc). Vùng sáng là nơi text "nhìn" vào.
- Bên dưới: từ nào trong caption được các patch ảnh chú ý nhiều (từ càng đậm càng được chú ý).

Cách đọc: nếu heatmap gần như đều trên cả ảnh, hoặc luôn dồn vào một góc cố định mọi mẫu, thì attention chưa mang thông tin riêng từng ảnh. Đây là phân tích định tính, không dùng để chấm điểm. Ảnh lưu ở `reports/figures/phase5_attention_maps.png`.

In [ ]:
import torch
from IPython.display import HTML
from PIL import Image

from vimmsd.analysis import interpretability as interp
from vimmsd.data.dataset import ViMMSDCollator, build_datasets
from vimmsd.models.classifier import build_model
from vimmsd.utils.checkpoint import load_checkpoint

try:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ATTN_RUN = "fusion_cross_attn_frozen"
    ckpt = OUT / run_name(ATTN_RUN, SEEDS[0]) / "best.pt"
    if not ckpt.exists():
        print(f"chưa có {ckpt}, bỏ qua cell này")
    else:
        cfg = load_config(f"configs/{ATTN_RUN}.yaml")
        model = build_model(cfg)
        load_checkpoint(ckpt, model, map_location=device)
        model.to(device).eval()
        collator = ViMMSDCollator(model.tokenizer, model.image_processor, cfg.data.max_length, cfg.data.get("max_caption_length"))
        labels = list(cfg.data.labels)
        test_ds = build_datasets(cfg, splits=("test",))["test"]
        idx = [i for i, r in enumerate(test_ds.records) if labels[r["label"]] in ("image-sarcasm", "multi-sarcasm")][:6]
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in collator([test_ds[i] for i in idx]).items()}
        attn = interp.cross_attention_maps(model, batch)
        preds = attn["logits"].argmax(-1).tolist()

        fig, axes = plt.subplots(len(idx), 2, figsize=(7, 3.4 * len(idx)))
        for row, i, p, a_map in zip(axes, idx, preds, attn["image_heatmap"]):
            rec = test_ds.records[i]
            img = Image.open(rec["image_path"]).convert("RGB")
            row[0].imshow(interp.encoder_view(img)); row[0].axis("off")
            row[0].set_title(f"thật: {labels[rec['label']]}\ndự đoán: {labels[p]}", fontsize=8)
            interp.overlay_heatmap(img, a_map, ax=row[1], title="attention text → ảnh")
        plt.tight_layout()
        Path("reports/figures").mkdir(parents=True, exist_ok=True)
        plt.savefig("reports/figures/phase5_attention_maps.png", dpi=150)
        plt.show()

        for j, i in enumerate(idx):
            n = int(batch["attention_mask"][j].sum())
            words, scores = interp.merge_subwords(model.tokenizer, batch["input_ids"][j, :n].tolist(),
                                                  attn["token_scores"][j, :n].tolist())
            print(f"[{labels[test_ds.records[i]['label']]}] dự đoán: {labels[preds[j]]}")
            display(HTML(interp.words_to_html(words, scores)))
        del model
        gc.collect()
        torch.cuda.empty_cache()
except Exception as e:  # cell này chỉ để minh hoạ, lỗi ở đây không được làm hỏng cả lần Save & Run All
    print("attention map lỗi, bỏ qua:", repr(e))


## Dòng kết quả cho `reports/results.md`

Copy các dòng dưới vào `reports/results.md` rồi commit. Bảng thống kê tài nguyên lấy cột `train (phút)`, `peak VRAM (GB)` và `trainable (M)` ở bảng tổng hợp.

In [ ]:
for seed in SEEDS:
    for name in RUNS:
        path = OUT / run_name(name, seed) / "results_row.md"
        if path.exists():
            print(path.read_text(encoding="utf-8").strip().splitlines()[-1])

## Lưu model (`best.pt` + `config.json`)

Mỗi run đã có sẵn trong `/kaggle/working/outputs/<run>/`: `best.pt` (checkpoint theo val macro-F1) và `config.json` (cấu hình + commit). Hai encoder đóng băng nên `best.pt` chỉ chứa khối cross-attention và head, vài chục MB; khi load, encoder được tải lại từ pretrained.
Cell dưới liệt kê các file cần giữ. Khuyến nghị giữ seed đầu tiên của 3 config cross-attention (đủ để tái lập và vẽ attention), các seed còn lại chỉ cần `test_metrics.json`.

In [ ]:
for name in RUNS:
    d = OUT / run_name(name, SEEDS[0])
    files = {f: (d / f).stat().st_size / 1e6 for f in ("best.pt", "config.json", "test_metrics.json") if (d / f).exists()}
    print(f"{d.name:<42}", {k: f"{v:.1f} MB" for k, v in files.items()})

## Đọc kết quả (nghiệm thu Phase 5)

Thứ tự đọc:
1. **Bảng tổng hợp:** `fusion_cross_attn_*_frozen` so với `fusion_concat_frozen` (cùng chế độ đóng băng) trên **macro-F1 3 lớp** và F1 từng lớp. Chỉ coi một chênh lệch là thật khi lớn hơn độ lệch chuẩn giữa các seed.
2. **Hai chiều so với nhau:** `t2i` (text attend ảnh) với `i2t` (ảnh attend text). Chiều nào hơn, và khác nhau chủ yếu ở lớp nào (nhất là `image-sarcasm` và `multi-sarcasm`).
3. **So với mốc một modality:** cross-attention có vượt `text_only_frozen` không. Concat ở Phase 4 đúng bằng số mẫu với text-only, cross-attention có tách khỏi mức đó không.
4. **Đường cong train:** early stop sớm với train loss tụt nhanh là overfit nhánh ảnh. Nếu loss dao động mạnh, thử `--override train.lr_head=3e-4` cho cả ba config rồi chạy lại.
5. **So sánh từng mẫu (cell "giúp / nhiễu"):** số mẫu cross-attention đúng mà mốc sai và ngược lại, theo từng nhãn.
6. **Attention map:** chỉ để mô tả định tính trong báo cáo, đừng dùng để kết luận mô hình "hiểu" ảnh.
7. Điền kết quả vào sheet: `Kết quả` của task 5.3 lấy từ `fusion_cross_attn_t2i_frozen`, 5.4 từ `fusion_cross_attn_i2t_frozen`; task 5.5 so với `fusion_concat_frozen`.